In [38]:
from pydantic_ai import Agent
from dotenv import load_dotenv
import os

load_dotenv()

#print(f"API: {os.getenv('GOOGLE_GEMINI_API_KEY')}")
agent = Agent(model="google-gla:gemini-2.5-flash")
              #,output_type=EmployeeModel) # 2.5-flash normally

result = await agent.run("Give me a programming joke")

result


AgentRunResult(output='Why do programmers prefer dark mode?\n\nBecause light attracts bugs!')

In [39]:
print(result.output)

Why do programmers prefer dark mode?

Because light attracts bugs!


In [40]:
result = await agent.run("Give me an IT employee working in Sweden, shortly")
result

AgentRunResult(output='**Lars Svensson, 34**\n\nA DevOps Engineer based in Stockholm, Sweden. Lars works for a fast-growing tech company, ensuring their cloud infrastructure (primarily AWS) runs smoothly. He\'s adept with Python scripting, Docker, Kubernetes, and CI/CD pipelines. Known for his calm demeanor and love for a good "fika," outside of work he enjoys exploring the Stockholm archipelago in the summer and hitting the ski slopes during winter.')

In [41]:
print(result.output)

**Lars Svensson, 34**

A DevOps Engineer based in Stockholm, Sweden. Lars works for a fast-growing tech company, ensuring their cloud infrastructure (primarily AWS) runs smoothly. He's adept with Python scripting, Docker, Kubernetes, and CI/CD pipelines. Known for his calm demeanor and love for a good "fika," outside of work he enjoys exploring the Stockholm archipelago in the summer and hitting the ski slopes during winter.


In [34]:
with open("employee.md", "w") as file:
    file.write(str(result.output))

In [ ]:
from pydantic import BaseModel, Field

class EmployeeModel(BaseModel):
    name: str = Field(description="name of employee") 
    age: int = Field(description="age of employee", gt = 17, lt = 70)
    salary: int = Field(description="monthly salary of employee between 35000 and 70000", gt = 35000, lt=70000)
    role: str = Field(description="role of the employee, what he/she is working with")

# result = await agent.run("Give me an IT employee working in Sweden", output_type=EmployeeModel)
# result

In [46]:
result = await agent.run(
    f"""Extract employee information from this text. 
    If salary is not mentioned, estimate a reasonable monthly salary for this role in Stockholm.
    
    """,
    output_type=list[EmployeeModel]
)
result

AgentRunResult(output=[EmployeeModel(name='Alice', age=30, salary=50000, role='Software Engineer')])

In [47]:
result.output[0].model_dump

<bound method BaseModel.model_dump of EmployeeModel(name='Alice', age=30, salary=50000, role='Software Engineer')>

In [31]:
employee = result.output
employee

'Here is the extracted employee information with an estimated monthly salary:\n\n*   **Name:** Lars Svensson\n*   **Age:** 34\n*   **Role:** DevOps Engineer\n*   **Company:** GreenLoop Tech\n*   **Location:** Stockholm, Sweden\n*   **Key Responsibilities:** Automating deployments, managing cloud infrastructure on AWS, ensuring platform stability and scalability.\n*   **Personality/Work Style:** Calm, analytical approach, enjoys fika breaks, values strong work-life balance.\n*   **Monthly Salary (Estimated):** SEK 70,000\n\n**Reasoning for Salary Estimate:**\nA DevOps Engineer in Stockholm, especially with 34 years of age implying several years of experience and responsibilities like managing AWS cloud infrastructure and ensuring platform stability, typically earns a competitive salary. A range of SEK 65,000 - SEK 75,000 is common for this level of expertise in the current market, making SEK 70,000 a reasonable mid-point estimate.'

In [33]:
isinstance(employee, EmployeeModel), isinstance(employee, BaseModel)

(False, False)

In [50]:
result = await agent.run("""
                         Give me 5 employees in AI and data engineering field
                         Generate different roles, the more senior the role, the higher the salary
                         """,
                         output_type=list[EmployeeModel])

result

AgentRunResult(output=[EmployeeModel(name='Alice', age=28, salary=40000, role='Junior Data Engineer'), EmployeeModel(name='Bob', age=32, salary=50000, role='AI Engineer'), EmployeeModel(name='Charlie', age=35, salary=58000, role='Senior Data Scientist'), EmployeeModel(name='David', age=40, salary=65000, role='Lead AI Engineer'), EmployeeModel(name='Eve', age=45, salary=69999, role='Principal AI Architect')])

In [51]:
employees = [employee.model_dump() for employee in result.output]
employees[:3]

[{'name': 'Alice', 'age': 28, 'salary': 40000, 'role': 'Junior Data Engineer'},
 {'name': 'Bob', 'age': 32, 'salary': 50000, 'role': 'AI Engineer'},
 {'name': 'Charlie',
  'age': 35,
  'salary': 58000,
  'role': 'Senior Data Scientist'}]

In [ ]:
print(type(result.output))
print(result.output)

import json
data = json.loads(result.output)
employee  EmployeeModel(**data)

print(employee.name)
#employee.role

<class 'str'>
**Lars Svensson** is a 34-year-old **DevOps Engineer** working at GreenLoop Tech, a sustainable software company in **Stockholm**. He spends his days automating deployments, managing cloud infrastructure on AWS, and ensuring the stability and scalability of their platform. Lars is known for his calm, analytical approach and his love for a good **fika** break. Outside of work, he often heads out to explore Sweden's nature, valuing the strong work-life balance his job provides.


AttributeError: 'str' object has no attribute 'name'

In [21]:
print(f"Type: {type(result.output)}")
print(f"Content: {result.output}")

Type: <class 'str'>
Content: **Lars Svensson** is a 34-year-old **DevOps Engineer** working at GreenLoop Tech, a sustainable software company in **Stockholm**. He spends his days automating deployments, managing cloud infrastructure on AWS, and ensuring the stability and scalability of their platform. Lars is known for his calm, analytical approach and his love for a good **fika** break. Outside of work, he often heads out to explore Sweden's nature, valuing the strong work-life balance his job provides.


In [53]:
import pandas as pd

pd.DataFrame(employees)


,name,age,salary,role
0,Alice,28,40000,Junior Data Engineer
1,Bob,32,50000,AI Engineer
2,Charlie,35,58000,Senior Data Scientist
3,David,40,65000,Lead AI Engineer
4,Eve,45,69999,Principal AI Architect
